In [ ]:
# Import section
from backtesting import Backtest
import pandas as pd
from backtesting import Strategy
from backtesting.lib import crossover
import numpy as np
import math

# META
MARGIN = 0.01
CASH = 10_000
COMMISSION = 0.00002
table_name = 'f0_raw'

# DEFINE
signal_arr = []

/home/chris/backtester/dema_strat/.venv/lib/python3.14/site-packages/backtesting/_plotting.py:55: UserWarning: Jupyter Notebook detected. Setting Bokeh output to notebook. This may not work in Jupyter clients without JavaScript support, such as old IDEs. Reset with `backtesting.set_bokeh_output(notebook=False)`.
  warnings.warn('Jupyter Notebook detected. '


Loading BokehJS ...

# Strategy

In [ ]:
# Define the EMA Strategy
class STRAT(Strategy):
    n1 = 9  # Fast EMA period
    n2 = 21  # Slow EMA period

    risk_pct = 0.01      # Risk 1% of account per trade
    risk_reward = 1      # TP = 2x the SL distance

    def init(self):
        super().init()

        # Calculate EMAs using Pandas directly
        close = pd.Series(self.data.Close)

        self.ema1 = self.I(lambda: close.ewm(span=self.n1, adjust=False).mean(), name="ema fast")
        self.ema2 = self.I(lambda: close.ewm(span=self.n2, adjust=False).mean(), name="ema slow")

    def next(self):
        price = self.data.Close[-1]

        prev_low  = self.data.Low[-2]
        prev_high = self.data.High[-2]

        long_signal  = crossover(self.ema1, self.ema2) 
        short_signal = crossover(self.ema2, self.ema1)

        risk_amt = max(self.equity, CASH) * self.risk_pct

        def get_size_by_leveraged(sl_dist):
            ideal_size = risk_amt / sl_dist
            # We subtract a buffer 85%
            max_leverage_size = (self.equity * 0.85) / (price * MARGIN)
            size = int(min(ideal_size, max_leverage_size))
            return size

        if long_signal:

            sl_dist = abs(price - prev_low)
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                signal_arr.append((self.data.index[-1], sl_dist, 0, 0))
                return

            size = get_size_by_leveraged(sl_dist)
            signal_arr.append((self.data.index[-1], sl_dist))

            if size <= 0:
                return

            self.position.close()
            self.buy(size=size, sl=price - sl_dist, tp=price + tp_dist)

        elif short_signal:

            sl_dist = abs(price - prev_high)
            tp_dist = sl_dist * self.risk_reward

            if sl_dist == 0:
                signal_arr.append((self.data.index[-1], sl_dist, 0, 0))
                return

            size = get_size_by_leveraged(sl_dist)
            signal_arr.append((self.data.index[-1], sl_dist))

            if size <= 0:
                return


            self.position.close()
            self.sell(size=size, sl=price + sl_dist, tp=price - tp_dist)


# Basic backtest

In [ ]:
# Load sample data and run a basic backtest
file_path = "~/market_data/XAUUSD/XAUUSD_M5.csv"

try:
  # Load data
  data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
  
  # Run backtest
  bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
  stats = bt.run()
  
  print(stats)
  print(stats['Profit Factor'])
  
except Exception as e:
  print(f"Error: {str(e)}")

Start                     2020-01-02 01:00:00
End                       2025-12-31 23:55:00
Duration                   2190 days 22:55:00
Exposure Time [%]                     1.50093
Equity Final [$]                      3.44698
Equity Peak [$]                   19033.35602
Commissions [$]                   17377.93402
Return [%]                          -99.96553
Buy & Hold Return [%]               184.03997
Return (Ann.) [%]                   -72.82941
Volatility (Ann.) [%]                22.74901
CAGR [%]                            -60.02928
Sharpe Ratio                         -3.20143
Sortino Ratio                        -1.09458
Calmar Ratio                         -0.72843
Alpha [%]                           -95.36401
Beta                                   -0.025
Max. Drawdown [%]                   -99.98189
Avg. Drawdown [%]                   -11.13351
Max. Drawdown Duration     2178 days 20:20:00
Avg. Drawdown Duration      146 days 01:28:00
# Trades                          

# SQLite connection

In [ ]:
import sqlite3
import os
from datetime import datetime

# Setup SQLite database
db_path = "backtest_results.db"
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Database setup complete at {db_path}")

conn.commit()

# Combination test

In [ ]:
from itertools import product
from collections import namedtuple


cursor.execute(f'''
CREATE TABLE IF NOT EXISTS {table_name} (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    asset TEXT NOT NULL,
    timeframe TEXT NOT NULL,
    n1 INTEGER NOT NULL,
    n2 INTEGER NOT NULL,
    risk_pct REAL NOT NULL,
    risk_reward REAL NOT NULL,
    return_pct REAL,
    max_drawdown_pct REAL,
    cagr_pct REAL,
    sharpe_ratio REAL,
    profit_factor REAL,
    win_rate_pct REAL,
    num_trades INTEGER,
    equity_final REAL,
    run_timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
)
''')

# Test all asset/timeframe combinations
assets = ['XAUUSD', 'EURUSD', 'GBPUSD', 'USDCAD', 'AUDUSD', 'USDCHF', 'US100', 'US500']
timeframes = ['M1', 'M5', 'M15', 'H1', 'H4', 'D1']

# Strategy parameters (from current STRAT class)
init_params = {
    'n1': 9,
    'n2': 21,
    'risk_pct': 0.01,
    'risk_reward': 1
}

parameters = [
    timeframes,
    assets,
    range(1, 6)  # rr
    [0.005, 0.01] # risk percent
]

total_combinations = math.prod(len(p) for p in parameters)
print(f"Testing {total_combinations} combinations...")

for timeframe, asset, rr, risk_pct in product(*parameters):
    file_path = f"~/market_data/{asset}/{asset}_{timeframe}.csv"
    
    try:
        # Load data
        data = pd.read_csv(file_path, parse_dates=['Datetime'], index_col='Datetime')
        
        # Run backtest
        bt = Backtest(data, STRAT, cash=CASH, commission=COMMISSION, margin=MARGIN)
        stats = bt.run(risk_reward=rr, risk_pct=risk_pct)
        
        # Extract metrics
        return_pct = stats['Return [%]']
        max_drawdown_pct = stats['Max. Drawdown [%]']
        cagr_pct = stats['CAGR [%]']
        sharpe_ratio = stats['Sharpe Ratio']
        profit_factor = stats['Profit Factor']
        win_rate_pct = stats['Win Rate [%]']
        num_trades = stats['# Trades']
        equity_final = stats['Equity Final [$]']
        
        # Store in database
        cursor.execute(f'''
            INSERT INTO {table_name} 
            (asset, timeframe, n1, n2, risk_pct, risk_reward, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', (asset, timeframe, init_params['n1'], init_params['n2'], risk_pct, rr, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final))
        
        conn.commit()
        
        print(f"✓ {asset}_{timeframe}: Return={return_pct:.2f}%, DD={max_drawdown_pct:.2f}%, Sharpe={sharpe_ratio:.2f}, WinRate={win_rate_pct:.2f}%")
        
    except Exception as e:
        print(f"✗ {asset}_{timeframe}: Error - {str(e)}")
        # Store error record
        cursor.execute(f'''
            INSERT INTO {table_name} 
            (asset, timeframe, n1, n2, risk_pct, risk_reward, return_pct, max_drawdown_pct, cagr_pct, sharpe_ratio, profit_factor, win_rate_pct, num_trades, equity_final)
            VALUES (?, ?, ?, ?, ?, ?, NULL, NULL, NULL, NULL, NULL, NULL, NULL, NULL)
        ''', (asset, timeframe, init_params['n1'], init_params['n2'], risk_pct, rr))
        conn.commit()

print("\nBacktesting complete!")
print(f"Results stored in {db_path}")

### Summary

In [ ]:
query_pf = f"""
SELECT asset, timeframe, risk_reward, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE profit_factor IS NOT NULL
ORDER BY profit_factor DESC
LIMIT 20
"""
top_pf = pd.read_sql_query(query_pf, conn)

query_wr = f"""
SELECT asset, timeframe, risk_reward, return_pct, profit_factor, win_rate_pct, num_trades, equity_final
FROM {table_name}
WHERE win_rate_pct IS NOT NULL
ORDER BY win_rate_pct DESC
LIMIT 20
"""
top_wr = pd.read_sql_query(query_wr, conn)

print("Top 20 by Profit Factor")
print(top_pf.to_string(index=False))
print(f"Average profit factor (top 20): {top_pf['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20 by profit factor): {top_pf['win_rate_pct'].mean():.2f}%")
print("\nTop 20 by Win Rate")
print(top_wr.to_string(index=False))
print(f"Average profit factor (top 20 by win rate): {top_wr['profit_factor'].mean():.4f}")
print(f"Average win rate (top 20): {top_wr['win_rate_pct'].mean():.2f}%")


In [ ]:
bt.plot(open_browser=False, filename=f"{table_name}.html", resample=False, plot_pl=False, plot_equity=False, plot_volume=False)